# Name That LLM — Test Notebook

यह notebook केवल data loading, validation और baseline testing के लिए है।
यह Kaggle पर कोई submission नहीं भेजती।

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/name-that-llm')
print('Files:', list(DATA_DIR.glob('*')))

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
print('train shape:', train.shape)
print('test shape:', test.shape)
print('train columns:', train.columns.tolist())
print('model counts:')
display(train['model'].value_counts())
display(train.head())

In [ ]:
def make_model():
    features = FeatureUnion([
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=180000)),
        ('char', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=2, sublinear_tf=True, max_features=220000)),
    ])
    return Pipeline([
        ('features', features),
        ('classifier', LinearSVC(C=2.0, class_weight='balanced')),
    ])

if 'prompt_id' in train.columns:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, valid_idx = next(splitter.split(train, groups=train['prompt_id']))
    tr, va = train.iloc[train_idx], train.iloc[valid_idx]
else:
    tr, va = train_test_split(train, test_size=0.2, random_state=42, stratify=train['model'])

model = make_model()
model.fit(tr['text'].fillna(''), tr['model'])
valid_pred = model.predict(va['text'].fillna(''))
print('accuracy:', round(accuracy_score(va['model'], valid_pred), 4))
print('balanced accuracy:', round(balanced_accuracy_score(va['model'], valid_pred), 4))
print(classification_report(va['model'], valid_pred, zero_division=0))

In [ ]:
# Final local test artifact only — this does NOT submit to Kaggle.
final_model = make_model()
final_model.fit(train['text'].fillna(''), train['model'])
test_pred = final_model.predict(test['text'].fillna(''))
submission = pd.DataFrame({'id': test['id'], 'model': test_pred})
output_path = Path('/kaggle/working/test_submission.parquet')
submission.to_parquet(output_path, index=False)
print('Created local test artifact:', output_path)
print('Rows:', len(submission))
display(submission.head())
assert list(submission.columns) == ['id', 'model']
assert len(submission) == len(test)
print('Format check passed. No Kaggle submission was made.')